1. 깃허브 레포지토리 클론

In [ ]:
!git clone https://github.com/yuji4/MedSeg3D-KO.git
%cd MedSeg3D-KO

2. 라이브러리 설치

In [ ]:
!pip install -r requirements.txt

3. 임포트 및 한국어 폰트

In [ ]:
import sys, os
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from deep_translator import GoogleTranslator

sys.path.insert(0, '/content/MedSeg3D-KO')

!apt-get install -y fonts-nanum -q
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print("완료")

4. 모델 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "GoodBaiBai88/M3D-LaMed-Phi-3-4B"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    model_max_length=512,
    padding_side="right",
    use_fast=False,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True
)
print("로드 성공")

5. 인터페이스 실행

In [ ]:
import sys
for key in list(sys.modules.keys()):
    if 'src' in key or 'app' in key:
        del sys.modules[key]

sys.path.insert(0, '/content/MedSeg3D-KO')

from src.inference.segmentation import SegmentationPipeline

pipeline2 = SegmentationPipeline()
pipeline2.model = model
pipeline2.tokenizer = tokenizer
pipeline2._device = next(model.parameters()).device

import app.gradio_app as app_module
app_module._pipeline = pipeline2

app_module.demo.queue()
app_module.demo.launch(share=True)